# Local LLM Engine Showdown: Ollama vs. llama.cpp vs. Transformers

**NAIRR Workshop 2026 — running the *same* model three different ways on a Jetstream2 CPU instance**

We run **one model — Qwen2.5 (1.5B by default) — across three popular inference engines** and benchmark them head-to-head on the same hardware. Everything runs on the **CPU**, so it works on any Jetstream2 flavor (no GPU required).

| Engine | What it is | Strength | Model format |
|--------|-----------|----------|--------------|
| **Ollama** | The easiest "it just works" server. One command to pull & run. | Simplicity, great defaults | GGUF (4-bit, managed) |
| **llama.cpp** | The C/C++ engine *under* Ollama. Maximum portability. | Lightweight, quantized, runs anywhere | GGUF (4-bit) |
| **Transformers** | Hugging Face's **reference** library — the standard way researchers load a model. | Universal, full precision, huge ecosystem | HF safetensors (full precision) |

### The key idea to teach
> **Same weights → the engine & format mainly change SPEED. Quantization mainly changes QUALITY.**
>
> Ollama and llama.cpp run an optimized **4-bit GGUF** (small and fast on CPU). Transformers runs the **full-precision** weights — the *reference* implementation everyone learns first, but heavier. So any *speed* gap is mostly the **engine/format**, and any *quality* gap is mostly **quantization**. We measure both.

### What we measure
- **Latency** — total wall-clock per request
- **TTFT** — time to first token (responsiveness)
- **Throughput** — tokens / second during generation
- **Load time** — how long each engine takes to load the model
- **RAM** — system memory before/after each model loads
- **Quality** — side-by-side answers so you can judge for yourself

## 1. Configuration

Everything you might want to change lives here. **The model is one variable** — the default `qwen2.5:1.5b` is small, fast, and a plain instruct model, so all three engines return clean, directly comparable text. Swap it for `qwen2.5:3b` for stronger answers (needs more disk/RAM).

Use **`TEST_MODE`** to choose how much to run: **`"single"`** (one prompt — fastest, for a short class), **`"simple"`** (3 prompts — a good live demo), or **`"complete"`** (all prompts — the full benchmark). Progress bars show how far along each engine is.

In [ ]:
# ============================ CONFIG ============================
# Model identity across the three engines (must be the SAME model!)
MODEL_OLLAMA = "qwen2.5:1.5b"                    # Ollama tag        -> GGUF, 4-bit
MODEL_HF     = "Qwen/Qwen2.5-1.5B-Instruct"      # HuggingFace repo  -> Transformers, full precision
GGUF_REPO    = "Qwen/Qwen2.5-1.5B-Instruct-GGUF" # GGUF repo         -> llama.cpp
GGUF_QUANT   = "q4_k_m"                          # quant to pick from the GGUF repo
# NOTE: Qwen2.5-1.5B is a plain instruct model (no hidden "thinking"), so every engine
# returns clean comparable answers. Small enough to fit the default 20 GB m3.quad disk
# with no volume. Bump to qwen2.5:3b on a bigger disk (MODEL_HF="Qwen/Qwen2.5-3B-Instruct",
# GGUF_REPO="Qwen/Qwen2.5-3B-Instruct-GGUF").

# ---- Test depth: how much to run ----
# "single"   = ONE prompt          -> fastest possible demo (very short class)
# "simple"   = 3 prompts, short     -> good live demo
# "complete" = all 7 prompts, long  -> the full benchmark
TEST_MODE = "single"

if TEST_MODE == "single":
    PROMPT_SUBSET = ["sky"]                          # one clear, quick prompt
    MAX_TOKENS    = 256
elif TEST_MODE == "simple":
    PROMPT_SUBSET = ["greeting", "sky", "primes"]    # fast + illustrative
    MAX_TOKENS    = 256
else:                                                # "complete"
    PROMPT_SUBSET = None                             # use every prompt
    MAX_TOKENS    = 512

# Generation settings (identical for every engine -> fair comparison)
TEMPERATURE = 0.7
TOP_P       = 0.9

# Stream each model's answer to the screen as it generates (great for a live demo).
SHOW_OUTPUT = True

# Set False on a re-run to skip the (slow) dependency install
RUN_INSTALL = True
# ===============================================================
print("Config loaded. Model:", MODEL_HF)
print("Test mode:", TEST_MODE, "| max tokens per answer:", MAX_TOKENS)

## 2. Environment check

A quick look at the machine. This demo is **CPU-only by design** — it runs the same on a `m3.quad` CPU instance or any other flavor, with no GPU drivers or CUDA setup to worry about.

In [ ]:
import platform, sys, os

print("Python :", sys.version.split()[0])
print("OS     :", platform.platform())
print("CPUs   :", os.cpu_count())
print("-" * 60)
print(">> CPU mode: Ollama + llama.cpp + Transformers (all on the CPU)")

## 3. Install the three engines

This is the slow cell (a few minutes — run once).

- **PyTorch (CPU build) + Transformers** — the reference engine, via pip.
- **llama-cpp-python** — **compiles a CPU-only build** (a few minutes) so it can't accidentally pull a CUDA-linked wheel.
- **Ollama** — via the official install script.

> Set `RUN_INSTALL = False` in the config cell to skip this on later re-runs.

In [ ]:
if RUN_INSTALL:
    import sys, subprocess, os
    def pip(*a):
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *a], check=True)

    print("[1/4] utilities ...")
    pip("requests", "pandas", "matplotlib", "huggingface_hub", "tqdm", "ipywidgets")

    print("[2/4] torch (CPU) + transformers ...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch",
                    "--index-url", "https://download.pytorch.org/whl/cpu"], check=True)
    pip("transformers", "accelerate")

    print("[3/4] llama-cpp-python (compiling a CPU build — a few minutes) ...")
    # Remove any prior build, then COMPILE a CPU-only build. (Prebuilt wheels can resolve to
    # a CUDA-linked build that then fails to import with "libcudart.so.12 not found".)
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "llama-cpp-python"], capture_output=True)
    subprocess.run("sudo apt-get update -y && sudo apt-get install -y build-essential cmake python3-dev",
                   shell=True)
    cpu_env = {**os.environ, "CMAKE_ARGS": "-DGGML_CUDA=off", "FORCE_CMAKE": "1"}
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-cache-dir",
                    "--no-binary=llama-cpp-python", "llama-cpp-python"], env=cpu_env, check=True)

    print("[4/4] ollama ...")
    subprocess.run("curl -fsSL https://ollama.com/install.sh | sh", shell=True, check=True)
    print("Done.")
else:
    print("Skipped install (RUN_INSTALL=False)")

## 4. The benchmark prompts — *designed to expose real differences*

Each prompt **stresses a different dimension**. Because it's the same model in three engines, the *quality* is similar — so the interesting differences are about **speed** and about **quality under quantization** (the 4-bit GGUF engines vs full-precision Transformers).

| # | Prompt | What it stresses |
|---|--------|------------------|
| `greeting` | "Hi, who are you?" | **Latency floor / TTFT** — tiny output measures pure engine overhead |
| `sky` | "Why is the sky blue?" | **Short explanation** — quick to eyeball quality |
| `primes` | "Write a prime sieve" | **Code generation** — quantization can introduce subtle bugs |
| `essay` | "~400-word photosynthesis explanation" | **Sustained throughput** — the cleanest tokens/sec signal |
| `reasoning` | Multi-step discount problem | **Reasoning under quantization** |
| `json` | "8 planets as strict JSON" | **Format adherence** |
| `longctx` | Long passage + a question | **Prompt-processing (prefill) speed** |

In [ ]:
_PASSAGE = (
    "The National Artificial Intelligence Research Resource (NAIRR) is a shared "
    "national infrastructure that provides researchers and educators access to "
    "computing, data, models, and training. A pilot launched in 2024 to connect "
    "the U.S. research community to resources such as Jetstream2, a cloud "
    "computing system. Jetstream2 offers virtual machines that can be shared "
    "across a class through a single allocation, with each student using their "
    "own ACCESS identity. The goal is to lower the barrier to AI research for "
    "institutions that lack large on-premise clusters, while keeping usage "
    "accountable through per-user credit tracking."
)

PROMPTS = {
    "greeting":  "Hi! In one short sentence, who are you?",
    "sky":       "Why is the sky blue? Explain in 3-4 sentences.",
    "primes":    ("Write an efficient Python script that finds all prime numbers "
                  "from 1 to 10,000 using the Sieve of Eratosthenes, and prints "
                  "how many primes it found. Only output the code."),
    "essay":     ("Write a clear, detailed ~400-word explanation of how "
                  "photosynthesis works, written for a high-school student."),
    "reasoning": ("A store offers 25% off an item, then takes an additional 10% "
                  "off the already-discounted price. What is the single overall "
                  "percentage discount off the original price? Show your reasoning "
                  "step by step, then give the final number."),
    "json":      ('Return ONLY a JSON array of the 8 planets of our solar system. '
                  'Each element must be an object with keys "name", '
                  '"diameter_km" (number), and "moons" (number). No prose.'),
    "longctx":   ("Read the passage below and answer in one sentence: according to "
                  "the text, how does Jetstream2 keep shared classroom usage "
                  f"accountable?\n\nPASSAGE:\n{_PASSAGE}"),
}
if PROMPT_SUBSET:
    PROMPTS = {k: PROMPTS[k] for k in PROMPT_SUBSET if k in PROMPTS}
print(f"[{TEST_MODE} mode] {len(PROMPTS)} prompts ready:", ", ".join(PROMPTS))
print("Tip: set TEST_MODE='complete' in the config cell for the full run.")

## 5. Shared helpers — metrics, RAM, prompt formatting

Every engine call returns the **same standardized record** so the final table is apples-to-apples. We also print **system RAM** at each model load/unload — watch these numbers rise and fall, and compare them against `btop`/`htop` running in a side terminal.

In [ ]:
import time, subprocess, gc, json
from tqdm.auto import tqdm   # progress bars

RESULTS = []   # every benchmark row lands here

def record(framework, name, prompt_toks, completion_toks, ttft_s, total_s, output):
    gen_s = max(total_s - (ttft_s or 0), 1e-6)
    tps   = completion_toks / gen_s if completion_toks else 0.0
    row = {
        "framework": framework, "prompt": name,
        "prompt_tokens": prompt_toks, "completion_tokens": completion_toks,
        "ttft_s": round(ttft_s, 3) if ttft_s is not None else None,
        "total_s": round(total_s, 3),
        "tokens_per_sec": round(tps, 1),
        "output": output,
    }
    # Replace any earlier row for this (engine, prompt) so re-runs don't duplicate.
    for i, r in enumerate(RESULTS):
        if r["framework"] == framework and r["prompt"] == name:
            RESULTS.pop(i); break
    RESULTS.append(row)
    short = output.replace("\n", " ")[:70]
    tqdm.write(f"  [{framework:12}] {name:10} {completion_toks:4d} tok  "
               f"{total_s:6.2f}s  {tps:6.1f} tok/s  | {short}")
    return row

# Qwen2.5 is a plain instruct model -- send the prompt as-is so every engine gets the same input.
def build_messages(prompt):
    return [{"role": "user", "content": prompt}]

def mem_report(label=""):
    """Print system RAM used/free (MB) from /proc/meminfo. Match against btop."""
    try:
        info = {}
        with open("/proc/meminfo") as f:
            for line in f:
                key, val = line.split(":", 1)
                info[key] = int(val.split()[0])          # values are in kB
        total = info["MemTotal"] / 1024
        avail = info.get("MemAvailable", info["MemFree"]) / 1024
        used = total - avail
        print(f"   [RAM] {label:26} used {used:6.0f} MB | free {avail:6.0f} MB "
              f"({used/total*100:.0f}% of {total:.0f} MB)")
    except Exception as e:
        print("   [RAM] unavailable:", e)

print("Helpers ready.")
mem_report("baseline (no model loaded)")

## 6. Engine 1 — Ollama  🦙

The "it just works" path. We start the server, pull the model, and hit its HTTP API.
Ollama's response conveniently **reports its own token counts and timings**, so the stats here come straight from the engine. We run it on the **CPU** for a clean, universal comparison.

In [ ]:
import requests, time, subprocess, os

OLLAMA_URL = "http://127.0.0.1:11434"
os.environ.setdefault("OLLAMA_HOST", "127.0.0.1:11434")

# CPU-only demo: hide any GPU so Ollama always uses its (reliable) CPU runner.
env = {**os.environ, "CUDA_VISIBLE_DEVICES": ""}
subprocess.Popen(["ollama", "serve"], env=env,
                 stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

for _ in range(60):                                   # wait until the server is up
    try:
        if requests.get(OLLAMA_URL + "/api/tags", timeout=2).ok:
            print("Ollama server is up."); break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError("Ollama server did not start")

print(f"Pulling {MODEL_OLLAMA} (first time downloads weights) ...")
subprocess.run(["ollama", "pull", MODEL_OLLAMA], check=True)
print("Model ready (Ollama on CPU).")

In [ ]:
def run_ollama(name, prompt):
    body = {
        "model": MODEL_OLLAMA,
        "messages": build_messages(prompt),
        "stream": True,                     # stream so we can show the answer live
        "options": {"temperature": TEMPERATURE, "top_p": TOP_P,
                    "num_predict": MAX_TOKENS},
    }
    if SHOW_OUTPUT:
        print(f"\n--- [ollama] {name} ---")
    t0 = time.time(); first = None; ttft = None; text = ""; p_tok = c_tok = 0
    with requests.post(OLLAMA_URL + "/api/chat", json=body, stream=True, timeout=600) as r:
        for line in r.iter_lines():
            if not line:
                continue
            obj = json.loads(line)
            piece = obj.get("message", {}).get("content", "")
            if piece:
                if first is None:
                    first = time.time(); ttft = first - t0   # real time-to-first-token
                text += piece
                if SHOW_OUTPUT:
                    print(piece, end="", flush=True)
            if obj.get("done"):
                p_tok = obj.get("prompt_eval_count", 0)
                c_tok = obj.get("eval_count", 0)
    total_s = time.time() - t0
    if SHOW_OUTPUT:
        print()
    return record("ollama", name, p_tok, c_tok, ttft, total_s, text)

print("Running Ollama benchmark:")
for nm, pr in tqdm(PROMPTS.items(), total=len(PROMPTS), desc="Ollama"):
    run_ollama(nm, pr)
mem_report("after Ollama benchmark")

In [ ]:
# Free Ollama's memory before the next engine (keep_alive=0 unloads it instantly).
mem_report("before Ollama unload")
requests.post(OLLAMA_URL + "/api/generate",
              json={"model": MODEL_OLLAMA, "keep_alive": 0}, timeout=30)
for _ in range(40):                          # poll until the model is actually gone
    if not requests.get(OLLAMA_URL + "/api/ps", timeout=5).json().get("models", []):
        break
    time.sleep(0.5)
print("Ollama model unloaded.")
mem_report("after Ollama unload")

## 7. Engine 2 — llama.cpp (via `llama-cpp-python`)  ⚙️

The lightweight C/C++ engine that powers Ollama under the hood. We load the **same GGUF** directly and run it on the CPU. We stream so we can measure **TTFT** precisely.

In [ ]:
from huggingface_hub import list_repo_files, hf_hub_download

# auto-pick the right quant file from the GGUF repo (robust to naming)
files = [f for f in list_repo_files(GGUF_REPO) if f.endswith(".gguf")]
match = [f for f in files if GGUF_QUANT.lower() in f.lower()]
gguf_name = (match or files)[0]
print("Selected GGUF:", gguf_name, "(from", len(files), "files)")
gguf_path = hf_hub_download(repo_id=GGUF_REPO, filename=gguf_name)
print("Downloaded to:", gguf_path)

In [ ]:
from llama_cpp import Llama

t0 = time.time()
llm_cpp = Llama(model_path=gguf_path, n_gpu_layers=0, n_ctx=4096,
                n_threads=os.cpu_count(), verbose=False)
LLAMACPP_LOAD_S = time.time() - t0
print(f"llama.cpp loaded in {LLAMACPP_LOAD_S:.1f}s (CPU)")
mem_report("after llama.cpp load")

In [ ]:
def run_llamacpp(name, prompt):
    msgs = build_messages(prompt)
    t0 = time.time(); first = None; text = ""
    stream = llm_cpp.create_chat_completion(
        messages=msgs, max_tokens=MAX_TOKENS,
        temperature=TEMPERATURE, top_p=TOP_P, stream=True)
    if SHOW_OUTPUT:
        print(f"\n--- [llama.cpp] {name} ---")
    for chunk in stream:
        delta = chunk["choices"][0]["delta"].get("content", "")
        if delta:
            if first is None:
                first = time.time()
            text += delta
            if SHOW_OUTPUT:
                print(delta, end="", flush=True)
    if SHOW_OUTPUT:
        print()
    total_s = time.time() - t0
    ttft = (first - t0) if first else None
    p_tok = len(llm_cpp.tokenize(prompt.encode()))
    c_tok = len(llm_cpp.tokenize(text.encode())) if text else 0
    return record("llama.cpp", name, p_tok, c_tok, ttft, total_s, text)

print("Running llama.cpp benchmark:")
for nm, pr in tqdm(PROMPTS.items(), total=len(PROMPTS), desc="llama.cpp"):
    run_llamacpp(nm, pr)
mem_report("after llama.cpp benchmark")

In [ ]:
# free llama.cpp memory before Transformers
mem_report("before llama.cpp unload")
del llm_cpp
gc.collect()
time.sleep(1)
print("llama.cpp unloaded.")
mem_report("after llama.cpp unload")

## 8. Engine 3 — Hugging Face Transformers  🤗

The **reference** library — the standard way researchers load and run a model. It runs the **full-precision** weights (not the 4-bit GGUF the other two use), so it's the *quality baseline*: correct, universal, huge ecosystem — but heavier and **slower on CPU**, which is exactly why optimized engines like llama.cpp and Ollama exist.

We stream tokens as they generate (via `TextIteratorStreamer`) so we can measure **TTFT** just like the others.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, TextIteratorStreamer
from threading import Thread

print(f"Loading {MODEL_HF} with Transformers (CPU, full precision) ...")
t0 = time.time()
hf_tok = AutoTokenizer.from_pretrained(MODEL_HF)
hf_model = AutoModelForCausalLM.from_pretrained(MODEL_HF, torch_dtype=torch.float32)
hf_model.eval()
TRANSFORMERS_LOAD_S = time.time() - t0
print(f"Transformers loaded in {TRANSFORMERS_LOAD_S:.1f}s (CPU)")
mem_report("after Transformers load")

In [ ]:
def run_transformers(name, prompt):
    inputs = hf_tok.apply_chat_template(build_messages(prompt), return_tensors="pt",
                                        add_generation_prompt=True)
    streamer = TextIteratorStreamer(hf_tok, skip_prompt=True, skip_special_tokens=True)
    kw = dict(input_ids=inputs, max_new_tokens=MAX_TOKENS, do_sample=True,
              temperature=TEMPERATURE, top_p=TOP_P, streamer=streamer,
              pad_token_id=hf_tok.eos_token_id)
    if SHOW_OUTPUT:
        print(f"\n--- [transformers] {name} ---")
    t0 = time.time(); first = None; text = ""
    th = Thread(target=hf_model.generate, kwargs=kw); th.start()
    for piece in streamer:
        if first is None:
            first = time.time()
        text += piece
        if SHOW_OUTPUT:
            print(piece, end="", flush=True)
    th.join()
    if SHOW_OUTPUT:
        print()
    total_s = time.time() - t0
    ttft = (first - t0) if first else None
    p_tok = int(inputs.shape[1])
    c_tok = len(hf_tok(text)["input_ids"]) if text else 0
    return record("transformers", name, p_tok, c_tok, ttft, total_s, text)

print("Running Transformers benchmark (the full-precision baseline — slower on CPU):")
for nm, pr in tqdm(PROMPTS.items(), total=len(PROMPTS), desc="Transformers"):
    run_transformers(nm, pr)
mem_report("after Transformers benchmark")

## 9. The comparison table 📊

Same model, same prompts, same CPU — three engines. Now we line them up.

In [ ]:
import pandas as pd
pd.set_option("display.max_colwidth", 50)

df = pd.DataFrame(RESULTS).drop_duplicates(subset=["framework", "prompt"], keep="last")

print("=== Average performance per engine ===")
summary = (df.groupby("framework")
             .agg(avg_tokens_per_sec=("tokens_per_sec", "mean"),
                  avg_ttft_s=("ttft_s", "mean"),
                  avg_total_s=("total_s", "mean"),
                  total_completion_tokens=("completion_tokens", "sum"))
             .round(2).sort_values("avg_tokens_per_sec", ascending=False))
_load = {"ollama": None,
         "llama.cpp": round(LLAMACPP_LOAD_S, 1),
         "transformers": round(TRANSFORMERS_LOAD_S, 1)}
summary["load_time_s"] = pd.Series(_load)
summary

In [ ]:
# tokens/sec for every prompt x engine (the headline grid)
print("=== Throughput (tokens/sec) by prompt ===")
pivot_tps = df.pivot_table(index="prompt", columns="framework",
                           values="tokens_per_sec", aggfunc="mean")
pivot_tps.loc["** AVG **"] = pivot_tps.mean()
pivot_tps.round(1)

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
summary["avg_tokens_per_sec"].plot.bar(ax=ax[0], color=["#4c9", "#49c", "#c49"])
ax[0].set_title("Avg throughput (higher = better)"); ax[0].set_ylabel("tokens / sec")
ax[0].tick_params(axis="x", rotation=0)

summary["avg_ttft_s"].plot.bar(ax=ax[1], color=["#4c9", "#49c", "#c49"])
ax[1].set_title("Avg time-to-first-token (lower = better)"); ax[1].set_ylabel("seconds")
ax[1].tick_params(axis="x", rotation=0)
plt.tight_layout(); plt.show()

## 10. Quality side-by-side 🔍

Remember: Ollama & llama.cpp run **4-bit** GGUF weights; Transformers runs **full precision**. Read the answers and look for places where the quantized engines drift — most likely on `reasoning`, `primes` (a subtle bug), or `json` (format slips).

In [ ]:
def show_prompt(name, limit=600):
    if name not in PROMPTS or not any(r["prompt"] == name for r in RESULTS):
        print(f"(prompt '{name}' was not run in TEST_MODE='{TEST_MODE}')")
        return
    print("=" * 90)
    print("PROMPT:", PROMPTS[name][:120], "...\n")
    for fw in ["ollama", "llama.cpp", "transformers"]:
        rows = [r for r in RESULTS if r["prompt"] == name and r["framework"] == fw]
        if not rows: continue
        r = rows[0]
        print(f"--- {fw}  ({r['completion_tokens']} tok, {r['tokens_per_sec']} tok/s) ---")
        print(r["output"][:limit].strip())
        print()

# Show answers for whichever prompts ran this session (adapts to TEST_MODE).
for _name in ["reasoning", "primes", "json", "sky", "greeting", "essay", "longctx"]:
    if _name in PROMPTS:
        show_prompt(_name)

## 11. How to read these results 🎓

**Speed (Sections 9–10)**
- **llama.cpp & Ollama** (4-bit GGUF) are usually **much faster on CPU** and use the **least RAM** — the quantized model is smaller, so each token is cheaper. This is the whole reason these tools exist.
- **Transformers** (full precision) is the **reference**: universal and correct, but **slowest on CPU** and heaviest in RAM. It's what you reach for when you need the full ecosystem (fine-tuning, custom code, any model on the Hub) — not raw inference speed.

**Quality (Section 10)**
- Answers are *mostly* similar because **it's the same model**. Where they differ, it's usually the **4-bit quantization** (Ollama/llama.cpp) vs **full precision** (Transformers) — watch `reasoning`, `primes`, and `json`.

**Pick-the-engine cheat sheet**

| You want... | Use |
|-------------|-----|
| Easiest setup, single user, prototyping | **Ollama** |
| Embedded / max portability / smallest footprint | **llama.cpp** |
| Full ecosystem, fine-tuning, any Hub model, full precision | **Transformers** |

### Try next
- Bump the model to `qwen2.5:3b` (+ matching `MODEL_HF` / `GGUF_REPO`) to see the size/speed/quality trade-off.
- Set `TEST_MODE="complete"` for all seven prompts and the full benchmark.
- Add your own prompt to `PROMPTS` and see which engine handles it best.

> 💡 **Done? Shelve or delete your instance** so it stops drawing credits.